In [1]:
using LowLevelFEM
using LinearAlgebra
using SparseArrays
using Test

spectral_path = joinpath(
    dirname(pathof(LowLevelFEM)),
    "eigen.jl"
)

@assert isfile(spectral_path) "Copy spectral.jl into the LowLevelFEM src directory first: $spectral_path"

Base.include(
    LowLevelFEM,
    spectral_path
)

using LowLevelFEM: spectralTransformation

In [2]:
structured_rect_mesh(lx=10)

mat = Material("body")
U = Field([mat], type=:VectorField, dim=2, fieldName=:u);

In [3]:
K = ∫(ε(U) ⋅ D(:PlaneStress, mat) ⋅ ε(U))
M = ∫(U ⋅ mat.ρ ⋅ U);

In [4]:
bc = BoundaryCondition("left", ux=0, uy=0)

BoundaryCondition("left", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0, :ux => 0))

In [5]:
solveEigen(Symmetric(K), Symmetric(M), solver=:direct)

([1.255336890529273, -2.327613366938404, -2.6879879807254734, 9.976753958142934e10, 6.775378057879274e11, 2.264829897568584e12], [0.0029526283682570517 -0.0044363036733314305 … -0.02081172117568359 0.026964732363181552; 4.1882048976362146e-11 0.05929859240598946 … 0.05515854938135891 -0.0526321578386447; … ; -0.001705202422951408 0.003549042938344304 … 0.016609228042899382 0.021446005608172385; 0.05123613874587024 -0.02854022032314248 … -0.050954696035987006 -0.04711016665869725])

In [6]:
solveEigen(Symmetric(K), Symmetric(M), support=[bc])

([2.6035461792842393e9, 9.376097657714362e10, 6.302018177385383e11, 6.512032800498861e11, 2.1534328737270894e12, 5.016371047742032e12], [0.0 0.0 … 0.0 0.0; 0.0 0.0 … 0.0 0.0; … ; 0.0032468489344887044 -0.010796806051464476 … 0.02122213620636269 -0.025127991064828104; -0.0586181328331573 0.054931847093384785 … -0.04727401748899227 0.04367897615930301])

In [7]:
λ, ϕ = solveEigen(
    Symmetric(K),
    Symmetric(M);
    support=[bc],
    n=6,
    shift=0.0,
    solver=:arpack,
    solveroptions=(ncv=40, maxiter=10000)
)

@show λ

λ = [2.6035461792842393e9, 9.37609765771441e10, 6.302018177385382e11, 6.512032800498855e11, 2.1534328737270876e12, 5.016371047742031e12]


6-element Vector{Float64}:
 2.6035461792842393e9
 9.37609765771441e10
 6.302018177385382e11
 6.512032800498855e11
 2.1534328737270876e12
 5.016371047742031e12

In [ ]:
using Arpack

sK = maximum(abs, diag(Kr))
sM = maximum(abs, diag(Mr))

λs, ϕr, nconv = Arpack.eigs(
    Symmetric(Kr / sK),
    Symmetric(Mr / sM);
    nev=6,
    sigma=0.0,
    which=:LM,
    ncv=40,
    maxiter=10000
)

λ = λs .* (sK / sM)

@show nconv
@show λ

f = sqrt.(real.(λ)) ./ (2π)
@show f

residuals = [
    norm(Kr * ϕr[:, i] - λ[i] * Mr * ϕr[:, i]) /
    (norm(Kr * ϕr[:, i]) + abs(λ[i]) * norm(Mr * ϕr[:, i]))
    for i in eachindex(λ)
]

@show residuals

LoadError: UndefVarError: `Kr` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [9]:
λ, ϕ = solveEigen(
    Symmetric(K),
    Symmetric(M);
    support=[bc],
    n=6,
    shift=0.0,
    which=:LM,
    solver=:arpack,
    solveroptions=(
        explicittransform=:none,
        ncv=40,
        maxiter=10000
    )
)

@show λ
@show sqrt.(real.(λ)) ./ (2π)

λ = [2.60354617928424e9, 9.376097657714384e10, 6.302018177385385e11, 6.512032800498851e11, 2.1534328737270847e12, 5.016371047742039e12]
sqrt.(real.(λ)) ./ (2π) = [8120.874037820148, 48733.90279430153, 126345.55206537896, 128433.5305969854, 233553.18862110891, 356463.4105158542]


6-element Vector{Float64}:
   8120.874037820148
  48733.90279430153
 126345.55206537896
 128433.5305969854
 233553.18862110891
 356463.4105158542

In [10]:
solveEigen(Symmetric(K), Symmetric(M), shift=1, which=:LM)

([-0.09786844977501663, -0.15806082920636677, -0.1926102980921145, 9.976753958279018e10, 6.775378057848717e11, 2.2648298975713193e12], [-0.0011824510436370473 0.003356385691467687 … 0.02081172117562058 0.026964732362798154; -0.02618313742899015 -0.05303205953510504 … -0.05515854938134899 -0.052632157838665455; … ; -0.0005114763461133254 -0.005813125918724861 … -0.016609228042870874 0.021446005608576575; -0.033563859101738376 0.04783256817701477 … 0.05095469603593168 -0.04711016665867203])

In [11]:
solveEigen(K, M, support=[bc])

(ComplexF64[2.6035461792990165e9 + 0.0im, 9.376097657708163e10 + 0.0im, 6.302018177384562e11 + 0.0im, 6.512032800498076e11 + 0.0im, 2.153432873727059e12 + 0.0im, 5.016371047742e12 + 0.0im], ComplexF64[0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; 0.0 + 0.0im 0.0 + 0.0im … 0.0 + 0.0im 0.0 + 0.0im; … ; -0.003246848934482623 + 0.0im -0.010796806051465855 + 0.0im … 0.02122213620636363 + 0.0im 0.025127991064827958 + 0.0im; 0.058618132833124 + 0.0im 0.05493184709341712 + 0.0im … -0.04727401748899364 + 0.0im -0.04367897615930414 + 0.0im])

In [12]:
P = LowLevelFEM.prepare_eigen_system(
    K, M;
    support=[bc]
)

Kr, Mr = P.A, P.B

@show size(Kr)
@show nnz(Kr), nnz(Mr)

@show norm(Kr - Kr')
@show norm(Mr - Mr')

@show extrema(diag(Kr))
@show extrema(diag(Mr))

@show isposdef(Symmetric(Kr))
@show isposdef(Symmetric(Mr))

size(Kr) = (2200, 2200)


(nnz(Kr), nnz(Mr)) = (36815, 18476)
norm(Kr - Kr') = 7.624083117896295e-10
norm(Mr - Mr') = 0.0
extrema(diag(Kr)) = (98901.09890109849, 395604.39560440095)
extrema(diag(Mr)) = (8.722222222222265e-12, 3.488888888888941e-11)
isposdef(Symmetric(Kr)) = true
isposdef(Symmetric(Mr)) = true


true